# Build Enriched Candidate Dataset

Combines two public resume datasets and extracts a rich feature schema for use in the HireIQ ranking and fairness pipeline.

**Input datasets:**
- `hiring_app/data/candidates_cleaned.csv` — 2,484 resumes (Kaggle: gauravduttakiit/resume-dataset)
- `hiring_app/data/UpdatedResumeDataSet.csv` — 962 resumes (Kaggle: jillanisofttech / HuggingFace: Sachinkelenjaguri)

**Output:** `hiring_app/data/enriched_candidates.csv`

**Run order:** Execute all cells top to bottom. After running, the output CSV is ready to use.

In [ ]:
import pandas as pd
import re
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

try:
    from tqdm.notebook import tqdm
except ImportError:
    tqdm = lambda x, **kw: x
    print("Tip: pip install tqdm  — adds a progress bar to the extraction loop")

CURRENT_YEAR = datetime.now().year
DATA_DIR = Path("hiring_app/data")
OUTPUT_PATH = DATA_DIR / "enriched_candidates.csv"

print(f"Current year : {CURRENT_YEAR}")
print(f"Output path  : {OUTPUT_PATH}")

## Step 1 — Load & Combine Datasets

Dataset 2 can be loaded automatically from HuggingFace (requires `pip install datasets`) or manually placed at `hiring_app/data/UpdatedResumeDataSet.csv`.

In [ ]:
# ── Dataset 1: original resume dataset (2,484 rows, 28 categories) ──────────
df1 = pd.read_csv(DATA_DIR / "candidates_cleaned.csv")
df1 = (
    df1[["ID", "Resume_str", "Category"]]
    .rename(columns={"ID": "id", "Resume_str": "raw_text"})
)

# ── Dataset 2: UpdatedResumeDataSet (962 rows, 25 categories) ───────────────
# Both UpdatedResumeDataSet.csv and UpdatedResumeDataSet1.csv are the same data
# (Kaggle and HuggingFace copies). We use one and deduplicate.
df2_a = pd.read_csv(DATA_DIR / "UpdatedResumeDataSet.csv").rename(columns={"Resume": "raw_text"})
df2_b = pd.read_csv(DATA_DIR / "UpdatedResumeDataSet1.csv").rename(columns={"Resume": "raw_text"})
df2_raw = pd.concat([df2_a, df2_b], ignore_index=True).drop_duplicates(subset=["raw_text"])
df2 = df2_raw[["Category", "raw_text"]].copy()
df2["id"] = range(int(df1["id"].max()) + 1, int(df1["id"].max()) + 1 + len(df2))

# ── Combine & deduplicate across both datasets ───────────────────────────────
df = pd.concat([df1, df2], ignore_index=True)
before = len(df)
df = df.drop_duplicates(subset=["raw_text"]).reset_index(drop=True)

print(f"Dataset 1 (candidates_cleaned)  : {len(df1):,} rows")
print(f"Dataset 2 (UpdatedResumeDataSet): {len(df2):,} rows (after internal dedup)")
print(f"Cross-dataset duplicates removed : {before - len(df)}")
print(f"Combined total                   : {len(df):,} rows")
print(f"\nCategory distribution:")
print(df["Category"].value_counts().to_string())

## Step 2 — Text Cleaning

In [ ]:
def clean_text(text: str) -> str:
    """Lowercase, strip special characters, normalize whitespace."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = " ".join(text.split())
    return text

## Step 3 — Extraction Functions

In [ ]:
# ── Skills — derived from frequency analysis of both datasets ───────────────

SKILL_KEYWORDS = [
    # Programming languages
    "python", "java", "javascript", "typescript", "c++", "c#", "php", "swift",
    "kotlin", "ruby", "scala", "matlab", "perl", "bash", "shell", "vba",
    # ML / AI
    "machine learning", "deep learning", "nlp", "natural language processing",
    "computer vision", "reinforcement learning", "tensorflow", "pytorch", "keras",
    "scikit-learn", "xgboost", "lightgbm", "spacy", "nltk", "bert", "gpt", "llm",
    "random forest", "regression", "classification", "clustering",
    "neural network", "hugging face", "opencv",
    # Data science libraries
    "pandas", "numpy", "scipy", "matplotlib", "seaborn", "plotly",
    # Databases
    "sql", "mysql", "postgresql", "mongodb", "oracle", "sqlite", "redis",
    "cassandra", "elasticsearch", "snowflake", "bigquery", "nosql",
    # Big data
    "spark", "hadoop", "hive", "kafka", "airflow", "databricks",
    # Visualisation / BI
    "tableau", "power bi", "excel", "looker",
    # Web / Cloud
    "html", "css", "react", "angular", "vue", "bootstrap", "node.js",
    "flask", "django", "fastapi", "spring boot", ".net",
    "aws", "azure", "gcp", "docker", "kubernetes", "terraform", "ansible",
    "jenkins", "ci/cd", "git", "rest api", "graphql", "microservices",
    "xml", "json", "sharepoint",
    # Testing / QA
    "selenium", "appium", "junit", "pytest", "testng", "qa",
    "quality assurance", "automation testing", "manual testing",
    "jira", "postman",
    # Infra / Security / Networking
    "linux", "unix", "windows server", "networking", "firewall", "dns",
    # ERP / Accounting tools
    "sap", "erp", "tally", "quickbooks",
    # Finance domain
    "accounting", "financial reporting", "auditing", "budgeting",
    "forecasting", "financial modeling", "investment", "taxation",
    # HR / Business
    "recruitment", "talent acquisition", "payroll", "onboarding",
    "performance management", "hris", "crm", "salesforce",
    # Design
    "figma", "adobe xd", "photoshop", "illustrator", "indesign",
    # Soft / cross-cutting
    "communication", "leadership", "teamwork", "problem solving",
    "project management", "agile", "scrum", "data analysis",
    "business analysis", "microsoft office", "presentation",
    # Blockchain
    "blockchain", "ethereum",
]

SKILL_SYNONYMS = {
    r"\bml\b": "machine learning",
    r"\bdl\b": "deep learning",
    r"\bjs\b": "javascript",
    r"\bts\b": "typescript",
    r"\bcv\b": "computer vision",
    r"\bai\b": "machine learning",
    r"\bk8s\b": "kubernetes",
}

def extract_skills(text: str) -> list:
    """Extract and normalize skills from cleaned text."""
    normalized = text
    for pattern, replacement in SKILL_SYNONYMS.items():
        normalized = re.sub(pattern, replacement, normalized)
    return sorted({skill for skill in SKILL_KEYWORDS if skill in normalized})

In [ ]:
# ── Certifications ─────────────────────────────────────────────────────────

CERT_MAP = {
    "aws": [
        "aws certified", "aws solutions architect", "aws developer",
        "aws sysops", "aws cloud practitioner", "amazon web services certified",
    ],
    "azure": [
        "azure certified", "microsoft certified azure",
        "az-900", "az-104", "az-204", "azure fundamentals",
    ],
    "gcp": [
        "google cloud certified", "gcp certified",
        "associate cloud engineer", "professional cloud architect",
    ],
    "pmp": ["pmp", "project management professional", "pmbok"],
    "cfa": ["cfa", "chartered financial analyst"],
    "cpa": ["cpa", "certified public accountant"],
    "scrum": [
        "certified scrum master", "csm", "safe agilist",
        "scrum master certified", "professional scrum master",
    ],
    "google ux": ["google ux design", "google design certificate"],
    "google data": ["google data analytics", "google it support"],
    "ibm": ["ibm certified", "ibm data science", "ibm ai engineering"],
    "meta": ["meta certified", "facebook blueprint", "meta social media marketing"],
    "comptia": ["comptia a+", "comptia security+", "comptia network+"],
    "cisco": ["ccna", "ccnp", "cisco certified"],
    "oracle": ["oracle certified", "oracle database certified"],
    "six sigma": ["six sigma", "lean six sigma", "black belt", "green belt"],
}

def extract_certifications(raw_text: str) -> list:
    """
    Two-pass extraction:
    Pass 1 — dictionary match on full text (catches certs listed without trigger word)
    Pass 2 — keyword trigger scan for 'certified/certification/certificate' context
    Returns normalized cert keys (e.g. ['aws', 'pmp']).
    """
    text_lower = raw_text.lower()
    found = set()

    # Pass 1: dictionary match
    for cert_key, patterns in CERT_MAP.items():
        for pat in patterns:
            if pat in text_lower:
                found.add(cert_key)
                break

    # Pass 2: check sentences containing cert trigger words for any remaining signals
    trigger = re.compile(r"certif(?:ied|ication|icate)", re.IGNORECASE)
    for sentence in re.split(r"[.\n]", raw_text):
        if trigger.search(sentence):
            s_lower = sentence.lower()
            for cert_key, patterns in CERT_MAP.items():
                if cert_key not in found:
                    for pat in patterns:
                        if pat in s_lower:
                            found.add(cert_key)
                            break

    return sorted(found)

In [ ]:
# ── Experience years ────────────────────────────────────────────────────────

def extract_experience_years(text: str) -> int | None:
    """
    Priority order:
    1. Explicit 'X years of experience' → take max
    2. Year range spans (e.g. 2016–2021) → sum durations
    3. None
    """
    # Priority 1: explicit mention
    explicit = re.findall(r"(\d+)\+?\s+years?\s+(?:of\s+)?experience", text)
    if explicit:
        vals = [int(x) for x in explicit if int(x) <= 50]
        if vals:
            return max(vals)

    # Priority 2: year ranges
    ranges = re.findall(
        r"\b((?:19|20)\d{2})\b[\s\S]{0,20}?\b((?:19|20)\d{2})\b", text
    )
    if ranges:
        total = sum(
            int(end) - int(start)
            for start, end in ranges
            if 0 < int(end) - int(start) <= 20
        )
        if total > 0:
            return min(total, 40)

    return None

In [ ]:
# ── Job Titles — derived from frequency analysis of both datasets ────────────

JOB_TITLE_KEYWORDS = [
    # High frequency in datasets
    "teacher", "professor", "educator",
    "chef",
    "public relations",
    "accountant",
    "project manager", "project coordinator",
    "advocate", "lawyer", "legal advisor",
    "sales manager",
    "java developer",
    "operations manager",
    "mechanical engineer", "civil engineer", "electrical engineer",
    "business analyst",
    "software engineer", "software developer",
    "recruiter",
    "graphic designer", "web designer",
    "construction manager",
    "etl developer",
    "marketing manager",
    "devops engineer",
    "financial analyst",
    "database administrator",
    "python developer",
    "hr manager", "hr specialist",
    "network engineer", "network security engineer",
    "hadoop developer",
    "product manager",
    "test engineer", "qa engineer", "automation tester",
    "blockchain developer",
    "fitness trainer",
    "data analyst", "data scientist", "data engineer",
    "web developer", "full stack developer", "backend developer", "frontend developer",
    "scrum master",
    "product designer",
    "copywriter", "content writer",
    "sap consultant", "sap developer",
    "solutions architect", "cloud architect",
    "research scientist",
    "machine learning engineer",
    "ui/ux designer",
    ".net developer",
    "mobile developer", "android developer", "ios developer",
    "site reliability engineer",
]

def extract_job_titles(text: str) -> list:
    """Extract job titles from cleaned text."""
    return sorted({title for title in JOB_TITLE_KEYWORDS if title in text})

In [ ]:
# ── Education ───────────────────────────────────────────────────────────────

DEGREE_PATTERNS = [
    ("phd",      r"\b(?:ph\.?d|doctorate|doctor of philosophy)\b"),
    ("mba",      r"\bmba\b"),
    ("master",   r"\b(?:master'?s?|m\.?tech|m\.?sc|m\.?e\.?|m\.?s\.?)\b"),
    ("bachelor", r"\b(?:bachelor'?s?|b\.?tech|b\.?e\.?|b\.?sc|b\.?s\.?|b\.?a\.?|undergraduate)\b"),
    ("diploma",  r"\b(?:diploma|associate'?s?|polytechnic)\b"),
]

def extract_education(text: str) -> dict:
    """
    Extract highest degree, field of study, and tier.
    Degree priority: phd > mba > master > bachelor > diploma.
    Field is inferred from context words immediately following the degree keyword.
    Tier is left None (not reliably extractable from resume text).
    """
    text_lower = text.lower()
    degree = None
    field = None

    for deg_label, pattern in DEGREE_PATTERNS:
        match = re.search(pattern, text_lower)
        if match:
            degree = deg_label
            # Extract field from up to 80 chars after the degree keyword
            post = text_lower[match.end(): match.end() + 80]
            post_clean = re.sub(r"\b(?:in|of|from|at|the|and|a|an|with|for)\b", " ", post)
            words = [w for w in post_clean.split() if len(w) > 2][:5]
            if words:
                field = " ".join(words[:3])
            break

    return {"degree": degree, "field": field, "tier": None}

In [ ]:
# ── Industries ──────────────────────────────────────────────────────────────

INDUSTRY_MAP = {
    "technology": [
        "software", "developer", "engineer", "programming", "tech", "it ", "computer",
    ],
    "data & analytics": [
        "data scientist", "data analyst", "analytics", "business intelligence",
        "machine learning", "big data",
    ],
    "finance": [
        "finance", "banking", "investment", "accounting", "audit", "cfa", "cpa",
        "trading", "equity", "portfolio",
    ],
    "healthcare": [
        "healthcare", "medical", "clinical", "hospital", "nursing",
        "pharma", "biotech", "patient",
    ],
    "human resources": [
        "hr ", "human resources", "recruitment", "talent", "payroll", "onboarding",
    ],
    "marketing": [
        "marketing", "advertising", "brand", "campaign", "seo", "social media",
    ],
    "sales": [
        "sales", "business development", "account manager", "revenue", "crm",
    ],
    "design": ["ux", "ui", "figma", "adobe", "creative", "graphic", "design"],
    "operations": [
        "operations", "supply chain", "logistics", "procurement", "manufacturing",
    ],
    "legal": ["legal", "advocate", "attorney", "litigation", "compliance", "law"],
    "education": [
        "teacher", "professor", "curriculum", "training", "academic", "tutor",
    ],
}

def extract_industries(category: str, text: str) -> list:
    """Infer industries from Category label and resume text signals."""
    combined = (str(category) + " " + text).lower()
    found = {
        industry
        for industry, signals in INDUSTRY_MAP.items()
        if any(sig in combined for sig in signals)
    }
    return sorted(found) if found else ["unknown"]

In [ ]:
# ── Seniority ───────────────────────────────────────────────────────────────

def extract_seniority(text: str, experience_years: int | None) -> str:
    """
    Levels: intern / junior / mid / senior / executive
    Explicit keyword takes priority over experience_years fallback.
    """
    t = text.lower()

    if re.search(r"\bintern(?:ship)?\b", t):
        return "intern"
    if re.search(r"\b(?:executive|\bvp\b|vice president|ceo|cto|cfo|chief|director)\b", t):
        return "executive"
    if re.search(r"\b(?:senior|\bsr\.?\b|lead|principal|\bstaff\b)\b", t):
        return "senior"
    if re.search(r"\b(?:junior|\bjr\.?\b|entry.level|associate|graduate trainee)\b", t):
        return "junior"
    if re.search(r"\bmid.level\b", t):
        return "mid"

    # Fallback: experience years
    if experience_years is not None:
        if experience_years <= 2:
            return "junior"
        if experience_years <= 6:
            return "mid"
        if experience_years <= 12:
            return "senior"
        return "executive"

    return "mid"  # safe default

In [ ]:
# ── Age inference ───────────────────────────────────────────────────────────
# Age is optional and noisy — null is always better than a bad guess.

def infer_age(text: str, experience_years: int | None) -> int | None:
    """
    Attempt order:
    1. Explicit 'age: XX' mention
    2. Date of birth (DOB / born YYYY)
    3. Earliest year in text → assume grad at 22
    4. Experience years fallback → 22 + experience
    Consistency-checked and bounds-checked before returning.
    """
    t = text.lower()
    age = None

    # 1. Explicit age
    m = re.search(r"\bage[:\s]+(\d{2})\b", t)
    if m:
        candidate = int(m.group(1))
        if 16 <= candidate <= 80:
            age = candidate

    # 2. DOB / born year
    if age is None:
        m = re.search(r"\b(?:dob|date of birth|born)[:\s/]+(\d{4})\b", t)
        if m:
            birth_year = int(m.group(1))
            if 1940 <= birth_year <= 2005:
                age = CURRENT_YEAR - birth_year

    # 3. Earliest year in resume → grad assumption
    if age is None:
        years = [int(y) for y in re.findall(r"\b((?:19|20)\d{2})\b", t)
                 if 1990 <= int(y) <= CURRENT_YEAR]
        if years:
            candidate = (CURRENT_YEAR - min(years)) + 22
            if 20 <= candidate <= 70:
                age = candidate

    # 4. Experience fallback (least reliable)
    if age is None and experience_years is not None:
        age = 22 + experience_years

    # Consistency check
    if age is not None and experience_years is not None:
        if age < experience_years + 16:
            return None

    # Final bounds
    if age is not None and not (16 <= age <= 80):
        return None

    return age

In [ ]:
# ── Career gap ──────────────────────────────────────────────────────────────

def extract_career_gap(text: str) -> float | None:
    """
    Best-effort: finds years present in resume, looks for gaps
    between consecutive entries that are 1–5 years wide.
    Returns total gap in years or None if too unreliable.
    """
    years = sorted({
        int(y)
        for y in re.findall(r"\b((?:19|20)\d{2})\b", text)
        if 1990 <= int(y) <= CURRENT_YEAR
    })

    if len(years) < 3:
        return None

    total_gap = sum(
        years[i] - years[i - 1]
        for i in range(1, len(years))
        if 1 < years[i] - years[i - 1] <= 5
    )
    return round(float(total_gap), 1) if total_gap > 0 else None

In [ ]:
# ── Derived features ────────────────────────────────────────────────────────

def compute_derived_features(record: dict) -> dict:
    """Compute skill_count, resume_length, learning_velocity, early_achiever."""
    certs = record.get("certifications", [])
    exp = record.get("experience_years")
    age = record.get("age")

    learning_velocity = round(len(certs) / ((exp or 0) + 1), 3)

    early_achiever = (
        # Young with solid experience
        (age is not None and age < 28 and exp is not None and exp >= 3)
        # OR high cert density even if age unknown
        or (exp is not None and exp >= 3 and len(certs) >= 2 and (age is None or age < 30))
    )

    return {
        "skill_count": len(record.get("skills", [])),
        "resume_length": len(record.get("cleaned_text", "")),
        "learning_velocity": learning_velocity,
        "early_achiever": bool(early_achiever),
    }

## Step 4 — Run Extraction Pipeline

Processes every resume through all extraction functions. On ~3,400 rows this takes roughly 2–4 minutes. Lists and dicts are serialized as JSON strings so they survive CSV round-trips cleanly.

In [ ]:
records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
    raw = str(row["raw_text"])
    cleaned = clean_text(raw)
    exp_years = extract_experience_years(cleaned)
    age = infer_age(cleaned, exp_years)

    record = {
        "id":               int(row["id"]),
        "category":         str(row.get("Category", "unknown")),
        "raw_text":         raw,
        "cleaned_text":     cleaned,
        "skills":           extract_skills(cleaned),
        "certifications":   extract_certifications(raw),
        "experience_years": exp_years,
        "job_titles":       extract_job_titles(cleaned),
        "industries":       extract_industries(row.get("Category", ""), cleaned),
        "education":        extract_education(cleaned),
        "seniority_level":  extract_seniority(cleaned, exp_years),
        "age":              age,
        "career_gap_years": extract_career_gap(cleaned),
    }
    record.update(compute_derived_features(record))
    records.append(record)

enriched = pd.DataFrame(records)

# Serialize list/dict columns to JSON strings for clean CSV storage
for col in ["skills", "certifications", "job_titles", "industries"]:
    enriched[col] = enriched[col].apply(json.dumps)
enriched["education"] = enriched["education"].apply(json.dumps)

print(f"Shape: {enriched.shape}")
enriched.head(3)

## Step 5 — Validation & Quality Check

In [ ]:
print("=" * 50)
print(" Dataset Overview")
print("=" * 50)
print(f"Total records  : {len(enriched):,}")
print(f"Columns        : {len(enriched.columns)}")

print("\n── Null / Missing Rates ──")
for col in ["experience_years", "age", "career_gap_years"]:
    null_pct = enriched[col].isna().mean() * 100
    print(f"  {col:<25} {null_pct:5.1f}% null")

print("\n── Seniority Distribution ──")
print(enriched["seniority_level"].value_counts().to_string())

print("\n── Top 10 Categories ──")
print(enriched["category"].value_counts().head(10).to_string())

print(f"\n── Derived Features ──")
print(f"  Early achievers   : {enriched['early_achiever'].sum():,} ({enriched['early_achiever'].mean()*100:.1f}%)")
print(f"  Avg skill count   : {enriched['skill_count'].mean():.1f}")
print(f"  Avg learning vel. : {enriched['learning_velocity'].mean():.3f}")

# ── Charts ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

enriched["skill_count"].hist(ax=axes[0], bins=25, color="steelblue", edgecolor="white")
axes[0].set_title("Skill Count Distribution")
axes[0].set_xlabel("Skills per resume")

enriched["experience_years"].dropna().hist(
    ax=axes[1], bins=25, color="coral", edgecolor="white"
)
axes[1].set_title("Experience Years Distribution")
axes[1].set_xlabel("Years")

enriched["seniority_level"].value_counts().plot(
    kind="bar", ax=axes[2], color="mediumseagreen", edgecolor="white"
)
axes[2].set_title("Seniority Level Distribution")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ── Spot-check 3 random records ──────────────────────────────────────────────
print("=" * 50)
print(" Spot-check (3 random records)")
print("=" * 50)

for rec in records[:3]:
    print(f"\nID: {rec['id']}  |  Category: {rec['category']}")
    print(f"  skills            : {rec['skills'][:6]}")
    print(f"  certifications    : {rec['certifications']}")
    print(f"  experience_years  : {rec['experience_years']}")
    print(f"  job_titles        : {rec['job_titles']}")
    print(f"  education         : {rec['education']}")
    print(f"  seniority_level   : {rec['seniority_level']}")
    print(f"  age               : {rec['age']}")
    print(f"  career_gap_years  : {rec['career_gap_years']}")
    print(f"  skill_count       : {rec['skill_count']}")
    print(f"  resume_length     : {rec['resume_length']}")
    print(f"  learning_velocity : {rec['learning_velocity']}")
    print(f"  early_achiever    : {rec['early_achiever']}")
    print(f"  raw_text preview  : {rec['raw_text'][:120].strip()}...")

## Step 6 — Save

In [ ]:
enriched.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(enriched):,} records → {OUTPUT_PATH}")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1e6:.1f} MB")
print(f"\nColumns saved:")
for col in enriched.columns:
    print(f"  {col}")

## Reading the Output

List and dict columns are stored as JSON strings. To use them in other notebooks or scripts:

```python
import pandas as pd, json

df = pd.read_csv("hiring_app/data/enriched_candidates.csv")

# Deserialize JSON columns
for col in ["skills", "certifications", "job_titles", "industries"]:
    df[col] = df[col].apply(json.loads)
df["education"] = df["education"].apply(json.loads)
```